# Calibración empírica de transiciones para el ABM

Construye probabilidades trimestrales de transición desde **Empleo Informal** hacia Empleo Formal, Empleo Informal, Desocupación e Inactividad, usando una jerarquía de calibración:

1. `cluster_final × cuartil del IVLR ampliado`;
2. cuartil general del IVLR;
3. distribución general.

Las probabilidades son descriptivas, no causales. El IVLR se utiliza únicamente en `t` y no se recalibra con los destinos observados.


In [1]:
# 1. Configuración y rutas
import os
import platform
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(os.getenv(
    "TFM_PROJECT_ROOT",
    r"C:\Users\tomas\OneDrive\Master Granada\TFM",
)).expanduser().resolve()

PANEL_PATH = PROJECT_ROOT / "Salidas" / "06_transiciones" / "panel_estados_auditado.rds"
IVLR_PATH = PROJECT_ROOT / "Salidas" / "09_ivlr" / "base_ivlr_preliminar.parquet"
OUT_DIR = PROJECT_ROOT / "Salidas" / "11_calibracion_abm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert PANEL_PATH.exists(), f"Panel no encontrado: {PANEL_PATH}"
assert IVLR_PATH.exists(), f"IVLR no encontrado: {IVLR_PATH}"
print(PROJECT_ROOT, OUT_DIR, sep="\n")


C:\Users\tomas\OneDrive\Master Granada\TFM
C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\11_calibracion_abm


In [2]:
# 2. Lectura robusta del panel
def leer_rds(ruta: Path) -> pd.DataFrame:
    try:
        import pyreadr
        resultado = pyreadr.read_r(str(ruta.resolve()))
        objetos = list(resultado.values())
        if objetos and isinstance(objetos[0], pd.DataFrame):
            return objetos[0]
    except Exception as error:
        print(f"Aviso pyreadr: {type(error).__name__}")

    with tempfile.TemporaryDirectory() as td:
        temporal = Path(td) / "panel.parquet"
        codigo_r = (
            "suppressMessages(library(arrow)); "
            f"x <- readRDS({str(ruta.resolve())!r}); "
            f"write_parquet(as.data.frame(x), {str(temporal)!r})"
        )
        proceso = subprocess.run(
            ["Rscript", "-e", codigo_r],
            capture_output=True, text=True, check=False,
        )
        if proceso.returncode != 0 or not temporal.exists():
            raise RuntimeError(proceso.stderr[-2000:])
        return pd.read_parquet(temporal)

panel = leer_rds(PANEL_PATH)
ivlr = pd.read_parquet(IVLR_PATH)
print(f"Panel: {panel.shape} | IVLR: {ivlr.shape}")


Panel: (283246, 38) | IVLR: (80300, 12)


In [4]:
# 3. Universo analítico y enlace
ESTADO_T = "estado_laboral_clasificado_t0"
ESTADO_T1 = "estado_laboral_clasificado_t1"
ESTADOS_DESTINO = ["Empleo Formal", "Empleo Informal", "Desocupación", "Inactividad"]
CLAVES = ["id_persona", "ANO4", "TRIMESTRE"]

assert panel["consistencia"].eq(True).all()
panel["id_persona"] = (
    panel["CODUSU"].astype(str) + "_" +
    panel["NRO_HOGAR"].astype(str) + "_" +
    panel["COMPONENTE"].astype(str)
).astype("string").str.strip()

for col in ["ANO4", "TRIMESTRE", "ANO4_t1", "TRIMESTRE_t1"]:
    panel[col] = pd.to_numeric(panel[col], errors="coerce").astype("Int64")
for col in [ESTADO_T, ESTADO_T1]:
    panel[col] = panel[col].astype("string").str.strip()

mask_ventana = (
    (panel["ANO4"].eq(2023) & panel["TRIMESTRE"].eq(4)) |
    panel["ANO4"].eq(2024) |
    (panel["ANO4"].eq(2025) & panel["TRIMESTRE"].le(3))
)
# Las máscaras pueden contener NA (estado en t0 tiene faltantes); al combinarlas
# con & se debe rellenar con False para que no se produzca "boolean value of NA".
mask_origen_informal = panel[ESTADO_T].eq("Empleo Informal").fillna(False)
mask_destino_valido = panel[ESTADO_T1].isin(ESTADOS_DESTINO).fillna(False)

panel_origen = panel.loc[
    mask_ventana & mask_origen_informal & mask_destino_valido
].copy()

ivlr["id_persona"] = ivlr["id_persona"].astype("string").str.strip()
for col in ["ANO4", "TRIMESTRE"]:
    ivlr[col] = pd.to_numeric(ivlr[col], errors="raise").astype("Int64")

cols_ivlr = CLAVES + ["ivlr_ampliado", "ivlr_basico", "cluster_final"]
ivlr_merge = ivlr[cols_ivlr].copy()
assert ivlr_merge.duplicated(CLAVES).sum() == 0

panel_analitico = panel_origen.merge(
    ivlr_merge, on=CLAVES, how="left", validate="many_to_one", indicator=True
)
assert len(panel_analitico) == 27891
assert panel_analitico["_merge"].eq("both").all()
assert panel_analitico["ivlr_ampliado"].notna().sum() == 22764

panel_analitico["PONDERA_t0"] = pd.to_numeric(panel_analitico["PONDERA"], errors="coerce")
assert panel_analitico["PONDERA_t0"].notna().all()
assert panel_analitico["PONDERA_t0"].gt(0).all()
print(f"Enlaces: {len(panel_analitico):,}; con IVLR ampliado: 22,764")

Enlaces: 27,891; con IVLR ampliado: 22,764


In [5]:
# 4. Cuartiles trimestrales y cluster de calibración
ETIQUETAS_CUARTIL = ["Q1 menor vulnerabilidad", "Q2", "Q3", "Q4 mayor vulnerabilidad"]
panel_calibracion = panel_analitico.loc[panel_analitico["ivlr_ampliado"].notna()].copy()
panel_calibracion["cuartil_ivlr_ampliado"] = (
    panel_calibracion.groupby(["ANO4", "TRIMESTRE"])["ivlr_ampliado"]
    .transform(lambda s: pd.qcut(s.rank(method="first"), 4, labels=ETIQUETAS_CUARTIL))
)
panel_calibracion["cluster_calibracion"] = (
    pd.to_numeric(panel_calibracion["cluster_final"], errors="coerce")
    .astype("Int64").astype("string").fillna("Sin cluster")
)
assert len(panel_calibracion) == 22764
assert panel_calibracion["cuartil_ivlr_ampliado"].notna().all()
display(panel_calibracion["cuartil_ivlr_ampliado"].value_counts(sort=False))


cuartil_ivlr_ampliado
Q1 menor vulnerabilidad    5694
Q2                         5691
Q3                         5688
Q4 mayor vulnerabilidad    5691
Name: count, dtype: int64

In [6]:
# 5. Funciones de calibración
def clasificar_soporte(n):
    if n >= 100:
        return "directa"
    if n >= 30:
        return "con cautela"
    return "respaldo obligatorio"

def tabla_probabilidades(datos, grupos, nivel):
    conteos = (datos.groupby(grupos + [ESTADO_T1], observed=True, dropna=False)
        .agg(n=("id_persona", "size"), poblacion_ponderada=("PONDERA_t0", "sum"))
        .reset_index())
    totales = (datos.groupby(grupos, observed=True, dropna=False)
        .agg(n_celda=("id_persona", "size"),
             poblacion_ponderada_celda=("PONDERA_t0", "sum"))
        .reset_index())
    out = conteos.merge(totales, on=grupos, how="left", validate="many_to_one")
    out["probabilidad_no_ponderada"] = out["n"] / out["n_celda"]
    out["probabilidad_ponderada"] = out["poblacion_ponderada"] / out["poblacion_ponderada_celda"]
    out["nivel_calibracion"] = nivel
    out["soporte_celda"] = out["n_celda"].map(clasificar_soporte)
    return out

def validar_probabilidades(tabla, grupos):
    control = tabla.groupby(grupos, observed=True, dropna=False).agg(
        p_no_pond=("probabilidad_no_ponderada", "sum"),
        p_pond=("probabilidad_ponderada", "sum"),
    ).reset_index()
    assert np.allclose(control["p_no_pond"], 1.0, atol=1e-10)
    assert np.allclose(control["p_pond"], 1.0, atol=1e-10)


In [7]:
# 6. Probabilidades por cluster × cuartil, por cuartil y generales
GRUPOS_ESPECIFICOS = ["cluster_calibracion", "cuartil_ivlr_ampliado"]
prob_cluster_cuartil = tabla_probabilidades(panel_calibracion, GRUPOS_ESPECIFICOS, "cluster_cuartil")
validar_probabilidades(prob_cluster_cuartil, GRUPOS_ESPECIFICOS)

prob_cuartil = tabla_probabilidades(panel_calibracion, ["cuartil_ivlr_ampliado"], "cuartil_general")
validar_probabilidades(prob_cuartil, ["cuartil_ivlr_ampliado"])

prob_general_larga = (panel_calibracion.groupby(ESTADO_T1)
    .agg(n=("id_persona", "size"), poblacion_ponderada=("PONDERA_t0", "sum"))
    .reset_index())
prob_general_larga["n_celda"] = len(panel_calibracion)
prob_general_larga["poblacion_ponderada_celda"] = panel_calibracion["PONDERA_t0"].sum()
prob_general_larga["probabilidad_no_ponderada"] = prob_general_larga["n"] / len(panel_calibracion)
prob_general_larga["probabilidad_ponderada"] = (
    prob_general_larga["poblacion_ponderada"] / panel_calibracion["PONDERA_t0"].sum()
)
assert np.isclose(prob_general_larga["probabilidad_ponderada"].sum(), 1.0)

diagnostico_celdas = (prob_cluster_cuartil[
    GRUPOS_ESPECIFICOS + ["n_celda", "poblacion_ponderada_celda", "soporte_celda"]
].drop_duplicates().sort_values(GRUPOS_ESPECIFICOS).reset_index(drop=True))
diagnostico_celdas["usar_nivel"] = np.where(
    diagnostico_celdas["n_celda"].ge(100), "cluster_cuartil", "cuartil_general"
)

reglas_respaldo = pd.DataFrame([
    {"prioridad": 1, "condicion": "cluster y cuartil disponibles; n_celda >= 100",
     "nivel_utilizado": "cluster_cuartil"},
    {"prioridad": 2, "condicion": "celda cluster-cuartil con n_celda < 100",
     "nivel_utilizado": "cuartil_general"},
    {"prioridad": 3, "condicion": "cluster o cuartil no disponible",
     "nivel_utilizado": "general"},
])

display(diagnostico_celdas)
display(reglas_respaldo)


,cluster_calibracion,cuartil_ivlr_ampliado,n_celda,poblacion_ponderada_celda,soporte_celda,usar_nivel
0,1,Q1 menor vulnerabilidad,1343,698299.0,directa,cluster_cuartil
1,1,Q2,1476,702626.0,directa,cluster_cuartil
2,1,Q3,1715,879559.0,directa,cluster_cuartil
3,1,Q4 mayor vulnerabilidad,2227,1291634.0,directa,cluster_cuartil
4,2,Q1 menor vulnerabilidad,1886,1255980.0,directa,cluster_cuartil
5,2,Q2,1408,819608.0,directa,cluster_cuartil
6,2,Q3,1126,632138.0,directa,cluster_cuartil
7,2,Q4 mayor vulnerabilidad,886,524242.0,directa,cluster_cuartil
8,3,Q1 menor vulnerabilidad,675,440049.0,directa,cluster_cuartil
9,3,Q2,404,181086.0,directa,cluster_cuartil


,prioridad,condicion,nivel_utilizado
0,1,cluster y cuartil disponibles; n_celda >= 100,cluster_cuartil
1,2,celda cluster-cuartil con n_celda < 100,cuartil_general
2,3,cluster o cuartil no disponible,general


In [10]:
# 7. Tabla operativa ancha para el ABM
prob_operativa = prob_cluster_cuartil.pivot_table(
    index=GRUPOS_ESPECIFICOS + ["n_celda", "poblacion_ponderada_celda", "soporte_celda"],
    columns=ESTADO_T1,
    values="probabilidad_ponderada",
    fill_value=0,
    observed=True,
).reset_index()

renombre = {
    "Empleo Formal": "p_empleo_formal",
    "Empleo Informal": "p_empleo_informal",
    "Desocupación": "p_desocupacion",
    "Inactividad": "p_inactividad",
}
prob_operativa = prob_operativa.rename(columns=renombre)
for col in renombre.values():
    if col not in prob_operativa.columns:
        prob_operativa[col] = 0.0
prob_operativa["suma_probabilidades"] = prob_operativa[list(renombre.values())].sum(axis=1)
prob_operativa = prob_operativa.merge(
    diagnostico_celdas[GRUPOS_ESPECIFICOS + ["usar_nivel"]],
    on=GRUPOS_ESPECIFICOS, how="left", validate="one_to_one"
)
assert np.allclose(prob_operativa["suma_probabilidades"], 1.0, atol=1e-10)

# Aplicar materialmente la jerarquía de respaldo.
# Las celdas con n < 100 sustituyen sus probabilidades específicas
# por las probabilidades generales del cuartil.

prob_cuartil_operativa = (
    prob_cuartil
    .pivot_table(
        index="cuartil_ivlr_ampliado",
        columns=ESTADO_T1,
        values="probabilidad_ponderada",
        fill_value=0,
        observed=True,
    )
    .reset_index()
    .rename(columns={
        "Empleo Formal": "p_cuartil_empleo_formal",
        "Empleo Informal": "p_cuartil_empleo_informal",
        "Desocupación": "p_cuartil_desocupacion",
        "Inactividad": "p_cuartil_inactividad",
    })
)

prob_operativa = prob_operativa.merge(
    prob_cuartil_operativa,
    on="cuartil_ivlr_ampliado",
    how="left",
    validate="many_to_one",
)

mask_respaldo_cuartil = (
    prob_operativa["usar_nivel"]
    .eq("cuartil_general")
)

sustituciones = {
    "p_empleo_formal": "p_cuartil_empleo_formal",
    "p_empleo_informal": "p_cuartil_empleo_informal",
    "p_desocupacion": "p_cuartil_desocupacion",
    "p_inactividad": "p_cuartil_inactividad",
}

for destino, respaldo in sustituciones.items():
    prob_operativa.loc[
        mask_respaldo_cuartil,
        destino,
    ] = prob_operativa.loc[
        mask_respaldo_cuartil,
        respaldo,
    ]

prob_operativa["suma_probabilidades"] = (
    prob_operativa[
        [
            "p_empleo_formal",
            "p_empleo_informal",
            "p_desocupacion",
            "p_inactividad",
        ]
    ]
    .sum(axis=1)
)

assert np.allclose(
    prob_operativa["suma_probabilidades"],
    1.0,
    atol=1e-10,
)

assert (
    prob_operativa.loc[
        mask_respaldo_cuartil,
        "usar_nivel",
    ]
    .eq("cuartil_general")
    .all()
)

columnas_auxiliares = [
    "p_cuartil_empleo_formal",
    "p_cuartil_empleo_informal",
    "p_cuartil_desocupacion",
    "p_cuartil_inactividad",
]

prob_operativa = prob_operativa.drop(
    columns=columnas_auxiliares,
)

prob_operativa.to_csv(
    OUT_DIR / "probabilidades_operativas_abm.csv",
    index=False,
    encoding="utf-8-sig",
)

display(prob_operativa)

,cluster_calibracion,cuartil_ivlr_ampliado,n_celda,poblacion_ponderada_celda,soporte_celda,p_desocupacion,p_empleo_formal,p_empleo_informal,p_inactividad,suma_probabilidades,usar_nivel
0,1,Q1 menor vulnerabilidad,1343,698299.0,directa,0.027974,0.126573,0.778933,0.066520,1.0,cluster_cuartil
1,1,Q2,1476,702626.0,directa,0.034417,0.071192,0.786090,0.108302,1.0,cluster_cuartil
2,1,Q3,1715,879559.0,directa,0.052992,0.091563,0.682054,0.173390,1.0,cluster_cuartil
3,1,Q4 mayor vulnerabilidad,2227,1291634.0,directa,0.109425,0.045298,0.639781,0.205496,1.0,cluster_cuartil
4,2,Q1 menor vulnerabilidad,1886,1255980.0,directa,0.043080,0.232648,0.673282,0.050990,1.0,cluster_cuartil
5,2,Q2,1408,819608.0,directa,0.046356,0.119394,0.774613,0.059637,1.0,cluster_cuartil
6,2,Q3,1126,632138.0,directa,0.075488,0.082518,0.708643,0.133351,1.0,cluster_cuartil
7,2,Q4 mayor vulnerabilidad,886,524242.0,directa,0.106943,0.075131,0.679890,0.138035,1.0,cluster_cuartil
8,3,Q1 menor vulnerabilidad,675,440049.0,directa,0.013803,0.278760,0.670421,0.037016,1.0,cluster_cuartil
9,3,Q2,404,181086.0,directa,0.022950,0.170957,0.722541,0.083551,1.0,cluster_cuartil


In [11]:
# 8. Exportaciones y controles finales
prob_cluster_cuartil.to_csv(OUT_DIR / "probabilidades_transicion_cluster_cuartil.csv", index=False, encoding="utf-8-sig")
prob_cuartil.to_csv(OUT_DIR / "probabilidades_transicion_cuartil.csv", index=False, encoding="utf-8-sig")
prob_general_larga.to_csv(OUT_DIR / "probabilidades_transicion_generales.csv", index=False, encoding="utf-8-sig")
prob_operativa.to_csv(OUT_DIR / "probabilidades_operativas_abm.csv", index=False, encoding="utf-8-sig")
diagnostico_celdas.to_csv(OUT_DIR / "diagnostico_tamanos_celda_abm.csv", index=False, encoding="utf-8-sig")
reglas_respaldo.to_csv(OUT_DIR / "reglas_respaldo_calibracion.csv", index=False, encoding="utf-8-sig")

versiones = pd.DataFrame({
    "componente": ["Python", "interprete", "pandas", "numpy", "pyarrow", "sistema", "entorno_conda"],
    "version": [sys.version.split()[0], sys.executable, pd.__version__, np.__version__,
                pa.__version__, platform.system(), os.environ.get("CONDA_DEFAULT_ENV", "desconocido")],
})
versiones.to_csv(OUT_DIR / "versiones_entorno_calibracion_abm.csv", index=False, encoding="utf-8-sig")

texto = (
    "Se construyeron probabilidades empíricas de transición desde empleo informal "
    "según cluster estructural y cuartil trimestral del IVLR ampliado. Las probabilidades "
    "son descriptivas y no causales. Las celdas con menos de 100 observaciones utilizan "
    "como respaldo las probabilidades por cuartil; cuando no hay cluster o cuartil, se "
    "utilizan las probabilidades generales."
)
with open(OUT_DIR / "conclusiones_calibracion_abm.txt", "w", encoding="utf-8") as f:
    f.write(texto)

assert len(panel_calibracion) == 22764
assert prob_operativa["suma_probabilidades"].between(0.999999, 1.000001).all()
print("Controles finales superados.")
print(f"Archivos exportados en: {OUT_DIR}")


Controles finales superados.
Archivos exportados en: C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\11_calibracion_abm
